# 05a Multimodal Stacking: Additional Feature Experiments

**Question:** do the feature groups explored in `07` improve the full stacking architecture from `05`?

`07` screens image–text features with simple logistic regressions. Here, reproduce `05` on the current data and add phrase–image, name–face and expression–sentiment features to **level 2 only**. Keep the level-1 scores, training rows, model settings, class-weighting treatment and noisy-OR rule unchanged across comparisons. `05` and `07` are not modified.

Run from `notebooks/` with the project kernel after `07` has exported its groups and verified the face cache. Requires `02`, both `03` feature groups, `04` features/caches and `07` exports. No image loading, neural encoding or model download occurs here. The first run trains the level-1 probes; article-grouped out-of-fold scores and fold checkpoints are cached for reuse. Running `05` first is not required.

All reported evaluation is **validation only**. Test features can receive level-1 scores as in `05`, but test labels are not evaluated. Results remain pending until this notebook is run; the historical `05` figures are not treated as a newly reproduced baseline.


## 0. Setup


In [1]:
import sys
import time
import json
import hashlib
from pathlib import Path
from itertools import combinations
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from src.evaluation import binary_metrics, best_f1_threshold, combine_fake_score, detection_rate_by_type, prevalence_table
from src.feature_sets import ARTIFACT_FEATURES, feature_columns

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
FEATURES_DIR = PROJECT_ROOT / "data/features"
CLIP_CACHE = PROJECT_ROOT / "data/interim/clip"
CACHE_DIR = PROJECT_ROOT / "data/interim/multimodal_05a"
RESULTS_DIR = PROJECT_ROOT / "results/multimodal_05a"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SPLITS = ["train_sample", "validation", "test"]
TARGETS = ["image_fake", "text_fake"]
ALL_TARGETS = TARGETS + ["is_fake"]
RANDOM_STATE, N_FOLDS, BOOTSTRAPS = 4012, 5, 500
RUN_TEST = False
assert not RUN_TEST, "Final-test evaluation is intentionally deferred in this comparison notebook"
BASE_GROUPS = ["text", "image", "clip", "clip_face", "clip_emb"]
NEW_GROUPS = {"phrases": "mm_phrase", "names": "mm_name_face", "expression": "mm_expression"}
L1 = ["l1_face_max", "l1_face_mean", "l1_face_largest", "l1_image", "l1_caption", "l1_pair"]
timings, chosen_C = {}, {}


In [2]:
def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    temporary.replace(path)


## 1. Validate and Join Matching Inputs

Require unique row IDs, exact split coverage, unchanged row order and finite predictors. Join raw embeddings in their existing column order, matching `05`. Exclude the shortcut columns declared in `src/feature_sets.py` from baseline text/image features.

Recheck `07`'s generated face manifest against current face/caption files, row/caption ordering and bounding boxes before training. The face manipulation annotation forms training-face labels only; it is never a predictor. Article IDs determine grouped folds only.

**Reproducibility:** dependencies are pinned in `requirements.txt`; the notebook fingerprints the inputs it actually uses and saves these with its results. No previous experiment result or investigation JSON is required to train or compare models. Use matching data outputs from `02`, `03`, `04` and `07`.

For exact comparison with the historical `05` run, use the same feature exports. Regenerated GPT-2 fluency values can differ slightly with inference settings; this previously changed the fitted trees. The original shared text exports restored exact prediction agreement. Newly generated features remain valid inputs for a fresh, internally controlled baseline comparison; they are not silently treated as an identical historical run.


In [3]:
input_fingerprints = {}
new_columns = {}


def load_split(split):
    path = PROCESSED_DIR / f"{split}.parquet"
    frame = pd.read_parquet(path)
    assert frame.row_id.is_unique
    input_fingerprints[str(path.relative_to(PROJECT_ROOT))] = file_sha256(path)
    row_order = frame.row_id.tolist()
    for group in BASE_GROUPS + list(NEW_GROUPS.values()):
        path = FEATURES_DIR / f"{group}_{split}.parquet"
        table = pd.read_parquet(path)
        assert table.row_id.is_unique and set(table.row_id) == set(row_order), f"{group}/{split}: mismatched row IDs"
        overlap = (set(frame) & set(table)) - {"row_id"}
        assert not overlap, f"Unexpected overlapping columns: {overlap}"
        frame = frame.merge(table, on="row_id", how="left", validate="one_to_one", sort=False)
        assert frame.row_id.tolist() == row_order
        assert np.isfinite(table.drop(columns="row_id").to_numpy()).all(), f"{group}/{split}: nonfinite features"
        input_fingerprints[str(path.relative_to(PROJECT_ROOT))] = file_sha256(path)
        if group in NEW_GROUPS.values():
            cols = [c for c in table if c != "row_id"]
            assert all(c.startswith(group + "_") for c in cols), f"Unexpected fields in {group}"
            if group in new_columns:
                assert new_columns[group] == cols, f"Column order differs for {group}/{split}"
            new_columns[group] = cols
    return frame


data = {split: load_split(split) for split in SPLITS}

train, val = data["train_sample"], data["validation"]
assert not (set(train.id) & set(val.id)), "Training/validation article overlap"
assert len(set().union(*(set(d.row_id) for d in data.values()))) == sum(map(len, data.values()))
IMG_EMB = [c for c in train if c.startswith("clip_img_")]
TXT_EMB = [c for c in train if c.startswith("clip_txt_")]
assert len(IMG_EMB) == len(TXT_EMB) == 768
metadata_columns = ["row_id", "image", "text", "id", "mtcnn_boxes", "image_fake", "fake_image_box"]
all_rows = pd.concat([data[s][metadata_columns] for s in SPLITS], ignore_index=True)
images = all_rows.drop_duplicates("image")[["image", "id", "mtcnn_boxes"]].reset_index(drop=True)
n_faces = images.mtcnn_boxes.str.len().to_numpy(dtype=int)
assert (n_faces > 0).all(), "05's face-summary protocol requires detected faces for every image"
face_emb = np.load(CLIP_CACHE / "faces_vitb16.npy", mmap_mode="r", allow_pickle=False)
assert face_emb.shape == (int(n_faces.sum()), 512)

face_manifest = pd.DataFrame(np.asarray([b for boxes in images.mtcnn_boxes for b in boxes], dtype=float), columns=["x1", "y1", "x2", "y2"])
face_manifest.insert(0, "image", np.repeat(images.image.to_numpy(), n_faces))
face_manifest.insert(0, "face_index", np.arange(len(face_manifest)))
verified_path = PROJECT_ROOT / "data/interim/multimodal_07/verified_face_manifest.json"
verified = json.loads(verified_path.read_text())
assert verified["face_file"] == file_sha256(CLIP_CACHE / "faces_vitb16.npy")
assert verified["caption_file"] == file_sha256(CLIP_CACHE / "captions_vitb16.npy")
assert verified["face_order"] == hashlib.sha256(pd.util.hash_pandas_object(face_manifest, index=False).values.tobytes()).hexdigest()
assert verified["row_order"] == fingerprint(all_rows.row_id.tolist())
assert verified["caption_order"] == fingerprint(all_rows.text.unique().tolist())
assert pd.read_parquet(verified_path.with_name("verified_face_index.parquet")).equals(face_manifest)
assert verified["reference_scores"] == {s: input_fingerprints[f"data/features/clip_face_{s}.parquet"] for s in SPLITS}
for lo in range(0, len(face_emb), 20000):
    assert np.isfinite(face_emb[lo:lo + 20000]).all()
print({s: len(d) for s, d in data.items()}, f"| {len(face_emb):,} verified face vectors")


{'train_sample': 49943, 'validation': 22126, 'test': 50705} | 294,383 verified face vectors


## 2. Level 1: Reproduce Once, Reuse Across Every Experiment

Match `05`'s four scaled logistic-regression probes:

| Probe | Inputs | Training label | Scores passed to level 2 |
|---|---|---|---|
| Face | Individual face embedding | Edited face or not | Maximum, mean, largest-face probability |
| Image | Whole-image embedding | `image_fake` | Image probability |
| Caption | Caption embedding | `text_fake` | Caption probability |
| Pair | Image, caption and element-wise product | `text_fake` | Joint image–caption probability |

Training scores are **five-fold article-grouped out-of-fold predictions**: no scored article is included in that fold's fit. Validation/test scores use a full-training fit. Preserve `05`'s regularisation selection over `C=(0.001, 0.01, 0.1)` on the first training grouped split; this is not nested hyperparameter selection. Neither validation labels nor the new features select C.

Fold and completed-probe caches carry input/code/configuration fingerprints and row-order checks. All feature experiments reuse exactly these same six scores. This can be the slowest section on its first run; it performs CPU classifier fitting, not CLIP extraction.


In [4]:
def make_probe(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))


def choose_C(X_train, y_train, groups, candidates=(0.001, 0.01, 0.1)):
    """Pick the regularisation strength by grouped CV AUC on train (validation stays untouched)."""
    fit_idx, pred_idx = next(GroupKFold(n_splits=N_FOLDS).split(X_train, y_train, groups))
    aucs = {}
    for C in candidates:
        model = make_probe(C).fit(X_train[fit_idx], y_train[fit_idx])
        aucs[C] = roc_auc_score(y_train[pred_idx], model.predict_proba(X_train[pred_idx])[:, 1])
    print("  CV AUC by C:", {c: round(a, 4) for c, a in aucs.items()})
    return max(aucs, key=aucs.get)


def embedding_matrix(frame, kind):
    # Same numerical columns/product as 05, avoiding unused pair arrays for single-modality probes.
    if kind == "image":
        return frame[IMG_EMB].to_numpy(np.float32)
    if kind == "caption":
        return frame[TXT_EMB].to_numpy(np.float32)
    image = frame[IMG_EMB].to_numpy(np.float32)
    caption = frame[TXT_EMB].to_numpy(np.float32)
    return np.hstack([image, caption, image * caption])


def atomic_arrays(path, **arrays):
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    temporary.replace(path)


def checked_scores(values, length):
    assert values.shape == (length,) and np.isfinite(values).all()
    assert ((values >= 0) & (values <= 1)).all()
    return values


def cached_oof_probe(X_train, y_train, groups, X_others, C, directory):
    oof = np.zeros(len(X_train))
    for fold, (fit_idx, pred_idx) in enumerate(GroupKFold(n_splits=N_FOLDS).split(X_train, y_train, groups)):
        assert not (set(groups[fit_idx]) & set(groups[pred_idx]))
        path = directory / f"fold_{fold}.npz"
        if path.exists():
            with np.load(path, allow_pickle=False) as saved:
                assert np.array_equal(saved["indices"], pred_idx), "Cached fold ordering differs"
                score = checked_scores(saved["score"], len(pred_idx))
        else:
            model = make_probe(C).fit(X_train[fit_idx], y_train[fit_idx])
            score = model.predict_proba(X_train[pred_idx])[:, 1]
            atomic_arrays(path, indices=pred_idx, score=score)
        oof[pred_idx] = score
        print(f"  article fold {fold + 1}/{N_FOLDS} complete", flush=True)
    full_path = directory / "full_training_predictions.npz"
    if full_path.exists():
        with np.load(full_path, allow_pickle=False) as saved:
            others = [checked_scores(saved[f"pred_{i}"], len(X)) for i, X in enumerate(X_others)]
    else:
        full = make_probe(C).fit(X_train, y_train)
        others = [full.predict_proba(X)[:, 1] for X in X_others]
        atomic_arrays(full_path, **{f"pred_{i}": p for i, p in enumerate(others)})
    return oof, others


def cached_probe(name, build_inputs, row_keys):
    directory = probe_dir / name
    directory.mkdir(parents=True, exist_ok=True)
    identity = {"stage1_key": stage1_key, "name": name, "row_keys": [fingerprint(keys) for keys in row_keys]}
    identity_path = directory / "identity.json"
    if identity_path.exists():
        assert json.loads(identity_path.read_text()) == identity, "Probe cache identity mismatch"
    else:
        write_json(identity_path, identity)
    path, C_path = directory / "completed.npz", directory / "selected_C.json"
    if path.exists():
        C = json.loads(C_path.read_text())["C"]
        with np.load(path, allow_pickle=False) as saved:
            oof = checked_scores(saved["oof"], len(row_keys[0]))
            others = [checked_scores(saved[f"pred_{i}"], len(keys)) for i, keys in enumerate(row_keys[1:])]
        print(f"{name}: reused completed grouped OOF/full-training scores", flush=True)
    else:
        X, y, groups, X_others = build_inputs()
        assert len(X) == len(y) == len(groups) == len(row_keys[0])
        assert all(len(array) == len(keys) for array, keys in zip(X_others, row_keys[1:]))
        if C_path.exists():
            C = json.loads(C_path.read_text())["C"]
        else:
            C = choose_C(X, y, groups)
            write_json(C_path, {"C": C})
        assert C in (0.001, 0.01, 0.1)
        oof, others = cached_oof_probe(X, y, groups, X_others, C, directory)
        atomic_arrays(path, oof=oof, **{f"pred_{i}": p for i, p in enumerate(others)})
    assert C in (0.001, 0.01, 0.1), "Invalid cached regularisation setting"
    chosen_C[name] = C
    return oof, others


reference = json.loads((PROJECT_ROOT / "notebooks/05_multimodal_stacking.ipynb").read_text())
reference_stage1 = ["".join(reference["cells"][i]["source"]) for i in [5, 7, 8, 10]]
own_sources = ["".join(c["source"]) for c in json.loads((PROJECT_ROOT / "notebooks/05a_multimodal_experiments.ipynb").read_text())["cells"]]
own_stage1 = [source for source in own_sources if "def make_probe(" in source or "def face_inputs(" in source]
stage1_config = {
    "schema": 1, "reference_stage1": fingerprint(reference_stage1), "implementation": fingerprint(own_stage1),
    "data": {k: v for k, v in input_fingerprints.items() if "/processed/" in k or "/clip_emb_" in k},
    "faces": verified, "folds": N_FOLDS, "C_candidates": [0.001, 0.01, 0.1],
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
    "image_columns": IMG_EMB, "caption_columns": TXT_EMB,
}
stage1_key = fingerprint(stage1_config)
probe_dir = CACHE_DIR / "level1" / stage1_key[:16]
probe_dir.mkdir(parents=True, exist_ok=True)
write_json(probe_dir / "configuration.json", stage1_config)


In [5]:
start = time.perf_counter()
faces = pd.DataFrame({"image": np.repeat(images.image.to_numpy(), n_faces),
                      "id": np.repeat(images.id.to_numpy(), n_faces),
                      "box": [list(map(int, b)) for boxes in images.mtcnn_boxes for b in boxes]})
faces["area"] = faces.box.apply(lambda b: (b[2] - b[0]) * (b[3] - b[1]))
fake_box = (all_rows[all_rows.image_fake == 1].drop_duplicates("image").set_index("image").fake_image_box
            .apply(lambda box: list(map(int, box))))
faces["face_fake"] = [int(fake_box.get(image) == box) for image, box in zip(faces.image, faces.box)]
image_split = all_rows.drop_duplicates("image").set_index("image").row_id.str.rsplit("_", n=1).str[0]
faces["split"] = faces.image.map(image_split)
is_train_face = (faces.split == "train").to_numpy()


def face_inputs():
    return (face_emb[is_train_face], faces.loc[is_train_face, "face_fake"].to_numpy(),
            faces.loc[is_train_face, "id"].to_numpy(), [face_emb[~is_train_face]])


oof, others = cached_probe("face", face_inputs,
                          [np.flatnonzero(is_train_face).tolist(), np.flatnonzero(~is_train_face).tolist()])
faces["p_face"] = 0.
faces.loc[is_train_face, "p_face"] = oof
faces.loc[~is_train_face, "p_face"] = others[0]
per_image = faces.groupby("image").agg(l1_face_max=("p_face", "max"), l1_face_mean=("p_face", "mean"))
per_image["l1_face_largest"] = faces.loc[faces.groupby("image").area.idxmax()].set_index("image").p_face
for split in SPLITS:
    order = data[split].row_id.tolist()
    data[split] = data[split].merge(per_image, left_on="image", right_index=True, how="left", validate="many_to_one", sort=False)
    assert data[split].row_id.tolist() == order

LEVEL1 = {"l1_image": ("image", "image_fake"), "l1_caption": ("caption", "text_fake"), "l1_pair": ("pair", "text_fake")}
for name, (kind, target) in LEVEL1.items():
    def probe_inputs():
        return (embedding_matrix(data["train_sample"], kind), data["train_sample"][target].to_numpy(),
                data["train_sample"].id.to_numpy(), [embedding_matrix(data[s], kind) for s in SPLITS[1:]])
    oof, others = cached_probe(name, probe_inputs, [data[s].row_id.tolist() for s in SPLITS])
    data["train_sample"][name] = oof
    for split, score in zip(SPLITS[1:], others):
        data[split][name] = score
train, val = data["train_sample"], data["validation"]
assert all(np.isfinite(d[L1].to_numpy()).all() for d in data.values())
timings["level1_seconds_this_run"] = time.perf_counter() - start
display(pd.DataFrame({target: {c: roc_auc_score(val[target], val[c]) for c in L1} for target in ALL_TARGETS}).round(4))


face: reused completed grouped OOF/full-training scores


l1_image: reused completed grouped OOF/full-training scores


l1_caption: reused completed grouped OOF/full-training scores


l1_pair: reused completed grouped OOF/full-training scores


,image_fake,text_fake,is_fake
l1_face_max,0.8084,0.5032,0.7489
l1_face_mean,0.7915,0.5005,0.7352
l1_face_largest,0.7519,0.4990,0.7049
l1_image,0.7990,0.4874,0.7369
l1_caption,0.4852,0.7335,0.5814
l1_pair,0.4988,0.8880,0.6585


## 3. Level 2: Fixed Architecture, Additional Feature Groups

Reproduce `05`'s baseline columns and its two histogram gradient-boosting classifiers: learning rate 0.05, maximum 600 iterations, 31 leaves, L2=1, early stopping on an internal 10% training holdout, seed 4012. Preserve default class weighting and the existing noisy-OR rule. No classifier or threshold-combination tuning is mixed into the feature experiment.

Compare the reproduced baseline, each new group, all pairs and all three. Include matching controls: phrase/name counts and availability; expression visual-valence summaries and VADER sentiment without interaction terms. Expression's improvement over the baseline alone does not establish a cross-modal interaction benefit.

At identical rows/settings, a difference can be attributed to adding the selected group, subject to exploratory validation uncertainty. Level-1 scores stay fixed. Saved historical `05` predictions are checked separately, because their data/environment provenance may differ.


In [6]:
TEXT = feature_columns(train, "txt_")
IMAGE = feature_columns(train, "img_")
CROSS = [c for c in train if c.startswith("clip_sim") or c.startswith("clip_face_")]
BASELINE = TEXT + IMAGE + CROSS + L1
assert not (set(BASELINE) & set(ARTIFACT_FEATURES))
group_columns = {name: new_columns[group] for name, group in NEW_GROUPS.items()}
controls = {
    "phrases": [c for c in group_columns["phrases"] if c.endswith(("_count", "_missing"))],
    "names": [c for c in group_columns["names"] if c.endswith(("_count", "_available"))],
    "expression": [c for c in group_columns["expression"] if "_product_" not in c and "_opposite_" not in c],
}
experiments = {"05 reproduced": BASELINE}
for size in [1, 2, 3]:
    for groups in combinations(group_columns, size):
        experiments["05 + " + " + ".join(groups)] = BASELINE + [c for group in groups for c in group_columns[group]]
for group in group_columns:
    experiments[f"Control: {group}"] = BASELINE + controls[group]
for columns in experiments.values():
    assert len(columns) == len(set(columns))
    assert set(columns) <= set(BASELINE + [c for cols in group_columns.values() for c in cols])
    assert not (set(columns) & set(ARTIFACT_FEATURES))
    assert np.isfinite(train[columns].to_numpy()).all() and np.isfinite(val[columns].to_numpy()).all()
print({"TEXT": len(TEXT), "IMAGE": len(IMAGE), "CROSS": len(CROSS), "LEVEL1": len(L1), "baseline": len(BASELINE)})
display(pd.DataFrame([{"experiment": name, "n_features": len(cols)} for name, cols in experiments.items()]))


{'TEXT': 19, 'IMAGE': 30, 'CROSS': 8, 'LEVEL1': 6, 'baseline': 63}


,experiment,n_features
0,05 reproduced,63
1,05 + phrases,77
2,05 + names,70
3,05 + expression,78
4,05 + phrases + names,84
5,05 + phrases + expression,92
6,05 + names + expression,85
7,05 + phrases + names + expression,99
8,Control: phrases,67
9,Control: names,67


In [7]:
def make_gbm():
    return HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=600, max_leaf_nodes=31, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=30, random_state=RANDOM_STATE,
    )


def fit_level2(features):
    """Fit one model per target on train_sample. Returns models and validation probabilities."""
    models, preds = {}, {}
    for target in TARGETS:
        models[target] = make_gbm().fit(train[features], train[target])
        preds[target] = models[target].predict_proba(val[features])[:, 1]
    return models, preds


def genuine_fpr_threshold(score, genuine, rate=0.01):
    values = np.asarray(score)[np.asarray(genuine, dtype=bool)]
    assert len(values) and np.isfinite(values).all() and 0 <= rate < 1
    allowed = int(np.floor(rate * len(values)))
    # >= threshold excludes all ties at the cutoff, so the allowed count is never exceeded.
    cutoff = np.sort(values)[len(values) - allowed - 1]
    return float(np.nextafter(cutoff, np.inf))



def compare_saved_baseline(rows, predictions, path, tolerance=1e-10):
    """Optional historical diagnostic; model fitting never depends on a previous run."""
    scores = {**predictions, "is_fake": combine_fake_score(predictions["image_fake"], predictions["text_fake"])}
    if not path.exists():
        return pd.DataFrame([{"status": "Saved 05 predictions unavailable; current-run baseline used"}])
    historical = pd.read_parquet(path)
    required = {"row_id", "p_image_fake", "p_text_fake", "p_is_fake"}
    if not required <= set(historical) or not historical.row_id.is_unique or set(historical.row_id) != set(rows.row_id):
        return pd.DataFrame([{"status": "Saved 05 predictions have different keys/schema; current-run baseline used"}])
    historical = historical.set_index("row_id").loc[rows.row_id]
    checks = []
    for target, score in scores.items():
        expected = historical[f"p_{target}"].to_numpy()
        if not np.isfinite(expected).all():
            checks.append({"target": target, "status": "Saved scores nonfinite; historical comparison unavailable"})
            continue
        maximum = float(np.max(np.abs(score - expected)))
        matches = bool(np.allclose(score, expected, atol=tolerance, rtol=0))
        checks.append({"target": target, "max_absolute_difference": maximum,
                       "saved_prediction_AUC": roc_auc_score(rows[target], expected),
                       "current_AUC": roc_auc_score(rows[target], score),
                       "status": "matches saved 05 predictions" if matches else
                                 "differs from saved 05; use current-run baseline for comparisons"})
    return pd.DataFrame(checks)


notebook_file = PROJECT_ROOT / "notebooks/05a_multimodal_experiments.ipynb"
code_identity = [(c["cell_type"], "".join(c["source"])) for c in json.loads(notebook_file.read_text())["cells"]]
run_configuration = {"schema": 1, "code_sha256": fingerprint(code_identity), "stage1_key": stage1_key,
    "reference_05_sha256": file_sha256(PROJECT_ROOT / "notebooks/05_multimodal_stacking.ipynb"),
    "evaluation_helper_sha256": file_sha256(PROJECT_ROOT / "src/evaluation.py"),
    "feature_selection_sha256": file_sha256(PROJECT_ROOT / "src/feature_sets.py"),
    "data_sha256": input_fingerprints, "experiments": experiments, "seed": RANDOM_STATE,
    "level2_parameters": make_gbm().get_params(), "selected_level1_C": chosen_C,
    "fit_split": "train_sample", "evaluation_split": "validation", "test_evaluated": False,
    "bootstrap": {"resamples": BOOTSTRAPS, "unit": "article id", "paired": True, "refit": False},
    "overall_score": "1 - (1 - p_image_fake) * (1 - p_text_fake)",
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__}}
run_key = fingerprint(run_configuration)[:16]
prediction_dir = CACHE_DIR / "validation_predictions" / run_key
prediction_dir.mkdir(parents=True, exist_ok=True)
validation_scores, metric_rows, strict_rows, detection_rows, runtime_rows = {}, [], [], [], []
genuine = val.is_fake.to_numpy() == 0
for name, columns in experiments.items():
    start = time.perf_counter()
    path = prediction_dir / f"{fingerprint(name)[:12]}.npz"
    cached = path.exists()
    if cached:
        with np.load(path, allow_pickle=False) as saved:
            assert np.array_equal(saved["row_id"], val.row_id.to_numpy(dtype=str))
            predictions = {t: checked_scores(saved[t], len(val)) for t in TARGETS}
    else:
        models, predictions = fit_level2(columns)
        atomic_arrays(path, row_id=val.row_id.to_numpy(dtype=str), **predictions)
    scores = {**predictions, "is_fake": combine_fake_score(predictions["image_fake"], predictions["text_fake"])}
    validation_scores[name] = scores
    for target, score in scores.items():
        f1_threshold = best_f1_threshold(val[target], score)
        strict_threshold = genuine_fpr_threshold(score, genuine)
        metric_rows.append({"experiment": name, "target": target, "n_features": len(columns),
                            **binary_metrics(val[target], score, f1_threshold),
                            "genuine_FPR": float((score[genuine] >= f1_threshold).mean())})
        strict_rows.append({"experiment": name, "target": target,
                            **binary_metrics(val[target], score, strict_threshold),
                            "genuine_FPR": float((score[genuine] >= strict_threshold).mean())})
        assert strict_rows[-1]["genuine_FPR"] <= .01 + 1e-12
        if target == "is_fake":
            for label, threshold in [("validation_F1", f1_threshold), ("validation_genuine_FPR_1pct", strict_threshold)]:
                table = detection_rate_by_type(val, score, threshold).reset_index()
                table["experiment"], table["operating_point"] = name, label
                detection_rows.append(table)
    elapsed = time.perf_counter() - start
    runtime_rows.append({"experiment": name, "seconds_this_run": elapsed, "predictions_from_cache": cached})
    print(f"{name}: {elapsed:.1f}s; cached={cached}", flush=True)
results, strict_results = pd.DataFrame(metric_rows), pd.DataFrame(strict_rows)
by_type, runtimes = pd.concat(detection_rows, ignore_index=True), pd.DataFrame(runtime_rows)
display(results.round(4))
display(strict_results[["experiment", "target", "genuine_FPR", "Recall", "Precision", "F1"]].round(4))


05 reproduced: 13.6s; cached=False


05 + phrases: 16.3s; cached=False


05 + names: 12.6s; cached=False


05 + expression: 14.3s; cached=False


05 + phrases + names: 12.8s; cached=False


05 + phrases + expression: 14.2s; cached=False


... (3 outputs left out)

Control: names: 10.8s; cached=False


Control: expression: 13.5s; cached=False


,experiment,target,n_features,AUC,PR-AUC,EER,ACC,Precision,Recall,F1,threshold,genuine_FPR
0,05 reproduced,image_fake,63,0.8717,0.8954,0.2231,0.7661,0.7349,0.8857,0.8033,0.3908,0.3767
1,05 reproduced,text_fake,63,0.9102,0.8402,0.1756,0.8659,0.7729,0.6998,0.7345,0.4166,0.0745
2,05 reproduced,is_fake,63,0.8721,0.9350,0.2213,0.7850,0.7904,0.9190,0.8499,0.4042,0.4779
3,05 + phrases,image_fake,77,0.8722,0.8955,0.2240,0.7653,0.7325,0.8896,0.8035,0.3790,0.3835
4,05 + phrases,text_fake,77,0.9165,0.8507,0.1700,0.8678,0.7659,0.7223,0.7435,0.4009,0.0800
5,05 + phrases,is_fake,77,0.8744,0.9361,0.2187,0.7859,0.7891,0.9234,0.8510,0.3905,0.4839
6,05 + names,image_fake,70,0.8724,0.8960,0.2236,0.7682,0.7403,0.8784,0.8034,0.4013,0.3656
7,05 + names,text_fake,70,0.9115,0.8416,0.1751,0.8645,0.7646,0.7066,0.7345,0.4057,0.0768
8,05 + names,is_fake,70,0.8734,0.9356,0.2177,0.7840,0.7861,0.9258,0.8502,0.3881,0.4940
9,05 + expression,image_fake,78,0.8749,0.8980,0.2208,0.7694,0.7393,0.8841,0.8053,0.3925,0.3676


,experiment,target,genuine_FPR,Recall,Precision,F1
0,05 reproduced,image_fake,0.0099,0.3556,0.9761,0.5213
1,05 reproduced,text_fake,0.0099,0.4884,0.9468,0.6444
2,05 reproduced,is_fake,0.0099,0.4089,0.9878,0.5784
3,05 + phrases,image_fake,0.0099,0.3421,0.9752,0.5065
4,05 + phrases,text_fake,0.0099,0.5167,0.9428,0.6675
5,05 + phrases,is_fake,0.0099,0.4186,0.9881,0.5881
6,05 + names,image_fake,0.0099,0.3463,0.9768,0.5114
7,05 + names,text_fake,0.0099,0.5024,0.9424,0.6554
8,05 + names,is_fake,0.0099,0.4055,0.9877,0.5750
9,05 + expression,image_fake,0.0099,0.3515,0.9756,0.5168


## 4. Optional Historical Baseline Comparison

All feature experiments use the freshly fitted baseline above. If `05`'s saved validation predictions are available, align them by `row_id` and report differences within absolute tolerance `1e-10`. Missing historical predictions or different scores do not block training or evaluation; they are explicitly reported rather than labelled as exact historical reproduction.

This diagnostic does not load saved predictions into any model. Identical historical probabilities require identical feature values and numerical settings. Current-run feature comparisons remain controlled because all variants share the same inputs, splits, level-1 scores and learner settings.


In [8]:
saved_path = PROJECT_ROOT / "results/predictions/multimodal_stacking_validation.parquet"
reproduction = compare_saved_baseline(val, validation_scores["05 reproduced"], saved_path)
display(reproduction)


,target,max_absolute_difference,saved_prediction_AUC,current_AUC,status
0,image_fake,0.0,0.871744,0.871744,matches saved 05 predictions
1,text_fake,0.0,0.910194,0.910194,matches saved 05 predictions
2,is_fake,0.0,0.872110,0.872110,matches saved 05 predictions


## 5. Paired AUC Differences and Low-False-Positive Behaviour

For each target, compare AUC with the reproduced baseline and matching controls using **500 paired article-level bootstrap resamples**. Also compare all three groups with each leave-one-group-out combination: a group can help alone yet be redundant once others are present.

Percentile 95% intervals describe validation-sample uncertainty conditional on fitted models; no refitting or multiplicity adjustment is performed. AUC ranks continuous scores, while precision/recall/F1 use thresholds. The overall score remains noisy-OR, assuming independent image/text manipulation probabilities; its calibration is not established.

The strict operating point controls false positives among **genuine pairs**, not all negatives of each modality-specific target. Thresholds are chosen and reported on validation, not a held-out final test. Inspect ranking gains and strict recall together before choosing candidates.


In [9]:
def prepare_weighted_auc(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores)
    order = np.argsort(scores, kind="stable")
    starts = np.r_[0, np.flatnonzero(np.diff(scores[order])) + 1]
    return order, starts, labels[order].astype(float)


def weighted_auc(prepared, weights):
    order, starts, labels = prepared
    ordered = np.asarray(weights)[order]
    positive = np.add.reduceat(ordered * labels, starts)
    negative = np.add.reduceat(ordered * (1 - labels), starts)
    denominator = positive.sum() * negative.sum()
    if denominator == 0:
        return np.nan
    return float(np.sum(positive * (np.cumsum(negative) - 0.5 * negative)) / denominator)


comparisons = [(name, "05 reproduced") for name in experiments if name.startswith("05 + ")]
comparisons += [(f"05 + {group}", f"Control: {group}") for group in group_columns]
all_groups_name = "05 + phrases + names + expression"
comparisons += [(all_groups_name, "05 + " + " + ".join(g for g in group_columns if g != removed)) for removed in group_columns]
prepared = {(name, target): prepare_weighted_auc(val[target], score)
            for name, scores in validation_scores.items() for target, score in scores.items()}
codes, articles = pd.factorize(val.id, sort=False)
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_auc = {key: np.empty(BOOTSTRAPS) for key in prepared}
start = time.perf_counter()
for b in range(BOOTSTRAPS):
    weights = np.bincount(rng.integers(len(articles), size=len(articles)), minlength=len(articles))[codes]
    for key, item in prepared.items():
        bootstrap_auc[key][b] = weighted_auc(item, weights)
    if (b + 1) % 100 == 0:
        print(f"Paired article bootstrap: {b + 1}/{BOOTSTRAPS}", flush=True)
lookup = results.set_index(["experiment", "target"])
strict_lookup = strict_results.set_index(["experiment", "target"])
difference_rows = []
for experiment, reference_name in comparisons:
    for target in ALL_TARGETS:
        delta = bootstrap_auc[(experiment, target)] - bootstrap_auc[(reference_name, target)]
        valid = delta[np.isfinite(delta)]
        assert len(valid) >= .95 * BOOTSTRAPS
        lower, upper = np.quantile(valid, [.025, .975])
        difference_rows.append({"experiment": experiment, "reference": reference_name, "target": target,
            "AUC_difference": float(lookup.loc[(experiment, target), "AUC"] - lookup.loc[(reference_name, target), "AUC"]),
            "CI_low": float(lower), "CI_high": float(upper), "valid_resamples": len(valid),
            "recall_change_at_1pct_genuine_FPR": float(strict_lookup.loc[(experiment, target), "Recall"] - strict_lookup.loc[(reference_name, target), "Recall"])})
differences = pd.DataFrame(difference_rows)
timings["bootstrap_seconds"] = time.perf_counter() - start
display(differences.round(4))
display(by_type[by_type.operating_point == "validation_genuine_FPR_1pct"].round(4))


Paired article bootstrap: 100/500


Paired article bootstrap: 200/500


Paired article bootstrap: 300/500


Paired article bootstrap: 400/500


Paired article bootstrap: 500/500


,experiment,reference,target,AUC_difference,CI_low,CI_high,valid_resamples,recall_change_at_1pct_genuine_FPR
0,05 + phrases,05 reproduced,image_fake,0.0004,-0.0005,0.0013,500,-0.0135
1,05 + phrases,05 reproduced,text_fake,0.0063,0.0048,0.0080,500,0.0283
2,05 + phrases,05 reproduced,is_fake,0.0023,0.0013,0.0033,500,0.0097
3,05 + names,05 reproduced,image_fake,0.0007,-0.0001,0.0015,500,-0.0093
4,05 + names,05 reproduced,text_fake,0.0013,0.0004,0.0021,500,0.0140
5,05 + names,05 reproduced,is_fake,0.0012,0.0004,0.0021,500,-0.0034
6,05 + expression,05 reproduced,image_fake,0.0032,0.0021,0.0044,500,-0.0041
7,05 + expression,05 reproduced,text_fake,0.0004,-0.0007,0.0014,500,0.0126
8,05 + expression,05 reproduced,is_fake,0.0033,0.0023,0.0043,500,0.0117
9,05 + phrases + names,05 reproduced,image_fake,0.0006,-0.0002,0.0013,500,-0.0127


,fake_cls,flagged_rate,n,experiment,operating_point
9,orig,0.0099,7473,05 reproduced,validation_genuine_FPR_1pct
10,text_attribute,0.0461,825,05 reproduced,validation_genuine_FPR_1pct
11,face_attribute,0.3118,3999,05 reproduced,validation_genuine_FPR_1pct
12,face_swap,0.3225,4788,05 reproduced,validation_genuine_FPR_1pct
13,face_swap&text_attribute,0.4240,533,05 reproduced,validation_genuine_FPR_1pct
...,...,...,...,...,...
193,face_swap&text_attribute,0.4203,533,Control: expression,validation_genuine_FPR_1pct
194,face_attribute&text_attribute,0.4353,487,Control: expression,validation_genuine_FPR_1pct
195,text_swap,0.5554,1894,Control: expression,validation_genuine_FPR_1pct
196,face_swap&text_swap,0.7687,1163,Control: expression,validation_genuine_FPR_1pct


## 6. Near-Duplicate and Prevalence Checks

Repeat validation metrics after excluding articles whose maximum image similarity to training is at least 0.95, using `04`'s near-duplicate table. Keep each experiment's validation-selected thresholds fixed; do not retune after removing rows.

For overall detection, also estimate alert counts and precision at 0.1%, 1% and 5% fake prevalence using measured overall TPR/FPR. These are projections under unchanged class-conditional behaviour, not deployment measurements.


In [10]:
near_dup_path = CLIP_CACHE / "near_duplicates.parquet"
near_dups = pd.read_parquet(near_dup_path)
duplicate_ids = set(near_dups.loc[near_dups.max_sim_to_train_sample >= .95, "id"])
keep = ~val.id.isin(duplicate_ids).to_numpy()
assert keep.any()
robustness_rows, prevalence_rows = [], []
for experiment, scores in validation_scores.items():
    for target, score in scores.items():
        threshold = float(lookup.loc[(experiment, target), "threshold"])
        robustness_rows.append({"experiment": experiment, "target": target, "rows": int(keep.sum()),
                                **binary_metrics(val.loc[keep, target], score[keep], threshold)})
    for setting, threshold in [("validation_F1", float(lookup.loc[(experiment, "is_fake"), "threshold"])),
                               ("validation_genuine_FPR_1pct", float(strict_lookup.loc[(experiment, "is_fake"), "threshold"]))]:
        table = prevalence_table(val.is_fake, scores["is_fake"], threshold).reset_index()
        table["experiment"], table["operating_point"] = experiment, setting
        prevalence_rows.append(table)
robustness, prevalence = pd.DataFrame(robustness_rows), pd.concat(prevalence_rows, ignore_index=True)
print(f"Excluded {int((~keep).sum()):,}/{len(val):,} validation rows from near-duplicate articles")
display(robustness.round(4))
display(prevalence[(prevalence.fake_rate == .01) & (prevalence.operating_point == "validation_genuine_FPR_1pct")].round(3))


Excluded 383/22,126 validation rows from near-duplicate articles


,experiment,target,rows,AUC,PR-AUC,EER,ACC,Precision,Recall,F1,threshold
0,05 reproduced,image_fake,21743,0.8712,0.8946,0.2243,0.7648,0.7330,0.8860,0.8023,0.3908
1,05 reproduced,text_fake,21743,0.9098,0.8401,0.1757,0.8654,0.7731,0.6993,0.7344,0.4166
2,05 reproduced,is_fake,21743,0.8716,0.9348,0.2220,0.7842,0.7895,0.9193,0.8495,0.4042
3,05 + phrases,image_fake,21743,0.8716,0.8948,0.2244,0.7637,0.7304,0.8895,0.8021,0.3790
4,05 + phrases,text_fake,21743,0.9161,0.8505,0.1705,0.8677,0.7671,0.7216,0.7437,0.4009
5,05 + phrases,is_fake,21743,0.8737,0.9359,0.2189,0.7849,0.7882,0.9235,0.8505,0.3905
6,05 + names,image_fake,21743,0.8719,0.8953,0.2241,0.7669,0.7383,0.8784,0.8023,0.4013
7,05 + names,text_fake,21743,0.9111,0.8415,0.1756,0.8642,0.7651,0.7063,0.7345,0.4057
8,05 + names,is_fake,21743,0.8728,0.9353,0.2184,0.7831,0.7851,0.9260,0.8498,0.3881
9,05 + expression,image_fake,21743,0.8743,0.8971,0.2220,0.7680,0.7373,0.8842,0.8041,0.3925


,fake_rate,fakes_per_100k,alerts,fakes_caught,false_alarms,precision,recall,experiment,operating_point
4,0.01,1000.0,1389.256,408.926,980.329,0.294,0.409,05 reproduced,validation_genuine_FPR_1pct
10,0.01,1000.0,1398.947,418.617,980.329,0.299,0.419,05 + phrases,validation_genuine_FPR_1pct
16,0.01,1000.0,1385.843,405.514,980.329,0.293,0.406,05 + names,validation_genuine_FPR_1pct
22,0.01,1000.0,1400.994,420.665,980.329,0.300,0.421,05 + expression,validation_genuine_FPR_1pct
28,0.01,1000.0,1399.424,419.095,980.329,0.299,0.419,05 + phrases + names,validation_genuine_FPR_1pct
34,0.01,1000.0,1396.421,416.092,980.329,0.298,0.416,05 + phrases + expression,validation_genuine_FPR_1pct
40,0.01,1000.0,1394.442,414.113,980.329,0.297,0.414,05 + names + expression,validation_genuine_FPR_1pct
46,0.01,1000.0,1402.632,422.303,980.329,0.301,0.422,05 + phrases + names + expression,validation_genuine_FPR_1pct
52,0.01,1000.0,1385.024,404.695,980.329,0.292,0.405,Control: phrases,validation_genuine_FPR_1pct
58,0.01,1000.0,1386.799,406.470,980.329,0.293,0.406,Control: names,validation_genuine_FPR_1pct


## 7. Record Findings and Configurations

Record results for every variant and group. The automated status is an **AUC-based exploratory screen**, not a final model decision: a wholly positive paired interval supports further testing; an interval spanning zero is inconclusive; a wholly nonpositive interval does not support improvement for that target. Strict-recall changes remain visible because AUC gains can coexist with worse low-FPR detection.

Save compact metrics, thresholds, runtime, configuration/data/code fingerprints and completion status under `results/multimodal_05a/<run fingerprint>/`. Large fold caches and full validation predictions stay under ignored `data/interim/multimodal_05a/`. No historical predictions or results are overwritten.


In [11]:
findings = differences.copy()
findings["status"] = np.where(findings.CI_low > 0, "candidate", np.where(findings.CI_high <= 0, "not supported in this experiment", "inconclusive"))
display(findings.round(4))
display(runtimes.round(2))
run_dir = RESULTS_DIR / run_key
run_dir.mkdir(parents=True, exist_ok=True)
configuration = {**run_configuration, "near_duplicate_sha256": file_sha256(near_dup_path),
                 "historical_predictions_sha256": file_sha256(saved_path) if saved_path.exists() else None,
                 "thresholds": results[["experiment", "target", "threshold"]].to_dict("records"),
                 "strict_thresholds": strict_results[["experiment", "target", "threshold"]].to_dict("records")}
write_json(run_dir / "configuration.json", {**configuration, "execution_status": "incomplete_until_metrics_saved"})
tables = {"validation_metrics": results, "validation_1pct_genuine_fpr": strict_results,
          "manipulation_type_rates": by_type, "paired_auc_differences": differences,
          "near_duplicate_exclusion": robustness, "prevalence_projection": prevalence,
          "baseline_reproduction": reproduction, "runtime": runtimes, "findings": findings}
for name, table in tables.items():
    path = run_dir / f"{name}.csv"
    temporary = path.with_suffix(".tmp")
    table.to_csv(temporary, index=False)
    temporary.replace(path)
write_json(run_dir / "configuration.json", {**configuration, "execution_status": "complete_validation",
           "completed_at_utc": datetime.now(timezone.utc).isoformat(), "timings_seconds": timings})
print(f"Saved current-run validation results to {run_dir.relative_to(PROJECT_ROOT)}")
display(Markdown("**Interpretation:** compare new groups with the reproduced 05 baseline and their controls. "
    "Check marginal value in combinations, strict-FPR recall, manipulation types and near-duplicate robustness. "
    "A useful result in 07 does not guarantee improvement here. Freeze the final configuration and thresholds "
    "before a separate final-test evaluation."))


,experiment,reference,target,AUC_difference,CI_low,CI_high,valid_resamples,recall_change_at_1pct_genuine_FPR,status
0,05 + phrases,05 reproduced,image_fake,0.0004,-0.0005,0.0013,500,-0.0135,inconclusive
1,05 + phrases,05 reproduced,text_fake,0.0063,0.0048,0.0080,500,0.0283,candidate
2,05 + phrases,05 reproduced,is_fake,0.0023,0.0013,0.0033,500,0.0097,candidate
3,05 + names,05 reproduced,image_fake,0.0007,-0.0001,0.0015,500,-0.0093,inconclusive
4,05 + names,05 reproduced,text_fake,0.0013,0.0004,0.0021,500,0.0140,candidate
5,05 + names,05 reproduced,is_fake,0.0012,0.0004,0.0021,500,-0.0034,candidate
6,05 + expression,05 reproduced,image_fake,0.0032,0.0021,0.0044,500,-0.0041,candidate
7,05 + expression,05 reproduced,text_fake,0.0004,-0.0007,0.0014,500,0.0126,inconclusive
8,05 + expression,05 reproduced,is_fake,0.0033,0.0023,0.0043,500,0.0117,candidate
9,05 + phrases + names,05 reproduced,image_fake,0.0006,-0.0002,0.0013,500,-0.0127,inconclusive


,experiment,seconds_this_run,predictions_from_cache
0,05 reproduced,13.65,False
1,05 + phrases,16.25,False
2,05 + names,12.63,False
3,05 + expression,14.31,False
4,05 + phrases + names,12.81,False
5,05 + phrases + expression,14.25,False
6,05 + names + expression,14.88,False
7,05 + phrases + names + expression,16.87,False
8,Control: phrases,11.72,False
9,Control: names,10.80,False


Saved current-run validation results to results/multimodal_05a/90bfcd10c7ac5811


**Interpretation:** compare new groups with the reproduced 05 baseline and their controls. Check marginal value in combinations, strict-FPR recall, manipulation types and near-duplicate robustness. A useful result in 07 does not guarantee improvement here. Freeze the final configuration and thresholds before a separate final-test evaluation.

# Summary

- **07:** tests whether proposed image–text features add information to a simple fixed classifier.
- **05a:** tests whether the same groups improve the full existing stacking architecture, beyond its forensic/text features and six level-1 scores.
- **Predictions:** separate image/caption manipulation probabilities; noisy-OR gives the overall score. Manipulation technique labels are used for error analysis, not as predictor inputs or additional classification outputs.
- **Evidence:** all findings above come from this notebook's validation run. Until it executes, improvements are pending. Historical prediction agreement is reported when available; every feature comparison uses this run's own fitted baseline.
- **Next decision:** retain useful groups based on paired differences, controls and operating-point trade-offs. Preserve negative findings. Test evaluation remains disabled; `05` is untouched.
